<a href="https://colab.research.google.com/github/napaporn3401-dot/GE338-Lab-4/blob/main/Lab4_6606614748V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install geemap
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='ee-napaporn3401')



In [66]:
# จังหวัดตาก
tak = ee.FeatureCollection("FAO/GAUL/2015/level1") \
        .filter(ee.Filter.eq('ADM1_NAME', 'Tak'))

Map.centerObject(tak, 8)
Map.addLayer(tak, {}, "Tak Province")
Map

Map(bottom=29982.0, center=[16.71110815479393, 98.79476507220035], controls=(WidgetControl(options=['position'…

In [86]:
Map = geemap.Map()

dem = ee.Image("USGS/SRTMGL1_003").clip(tak)
slope = ee.Terrain.slope(dem)

Map.addLayer(dem, {'min':0, 'max':1000}, "Elevation")
Map.addLayer(slope, {'min':0, 'max':60}, "Slope")

In [88]:
rain = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
        .filterDate('2020-01-01', '2020-12-31') \
        .select("total_precipitation_sum") \
        .mean() \
        .clip(tak)
Map.addLayer(rain, {'min':0, 'max':0.01}, "Rainfall")

In [97]:
river = ee.Image("WWF/HydroSHEDS/15ACC").gt(100)
distance_river = river.fastDistanceTransform().sqrt()

# 🔥 กลับค่า (สำคัญมาก)
dist_n = ee.Image.constant(1).subtract(dist_n)

In [98]:
landcover = ee.Image("MODIS/006/MCD12Q1/2020_01_01") \
              .select("LC_Type1") \
              .clip(tak)

Map.addLayer(landcover, {}, "Land Cover")

In [99]:
def normalize(img):
    min_val = ee.Number(
        img.reduceRegion(
            reducer=ee.Reducer.min(),
            geometry=tak,
            scale=5000,
            maxPixels=1e13
        ).values().get(0)
    )

    max_val = ee.Number(
        img.reduceRegion(
            reducer=ee.Reducer.max(),
            geometry=tak,
            scale=5000,
            maxPixels=1e13
        ).values().get(0)
    )

    min_img = ee.Image.constant(min_val)
    max_img = ee.Image.constant(max_val)

    return img.subtract(min_img).divide(max_img.subtract(min_img))

In [100]:
rain_n = normalize(rain)
slope_n = normalize(slope)
elev_n = normalize(dem)
dist_n = normalize(distance_river)
lc_n = normalize(landcover)

In [106]:
flood_risk = (rain_n.multiply(0.30)
             .add(slope_n.multiply(0.25))
             .add(elev_n.multiply(0.20))
             .add(dist_n.multiply(0.15))
             .add(lc_n.multiply(0.10)))

In [107]:
Map.addLayer(flood_risk,
             {'min':0, 'max':1,
              'palette':['green','yellow','red']},
             "Flood Risk Raw")
Map

Map(bottom=4189.0, center=[82.81981761923886, 243.26801194045913], controls=(WidgetControl(options=['position'…

In [108]:
# STEP: Classification
risk_class = flood_risk.expression(
    "(b(0) < 0.25) ? 1" +
    ": (b(0) < 0.45) ? 2" +
    ": 3"
)
# แสดงผล
Map.addLayer(risk_class,
             {'min':1, 'max':3,
              'palette':['green','yellow','red']},
             "Risk Class")
Map

Map(bottom=29942.0, center=[16.920194650443886, 458.2694800856983], controls=(WidgetControl(options=['position…

In [75]:
rain_n = normalize(rain)
Map.addLayer(rain_n, {'min':0, 'max':1}, "Rain Norm")

In [77]:
flood_low = (rain_n.multiply(0.24)
            .add(slope_n.multiply(0.25))
            .add(elev_n.multiply(0.20))
            .add(dist_n.multiply(0.15))
            .add(lc_n.multiply(0.10)))

In [78]:
flood_high = (rain_n.multiply(0.36)
             .add(slope_n.multiply(0.25))
             .add(elev_n.multiply(0.20))
             .add(dist_n.multiply(0.15))
             .add(lc_n.multiply(0.10)))

In [80]:
task = ee.batch.Export.image.toDrive(
    image=flood_risk,
    description='FloodRisk_Tak',
    folder='GEE',
    fileNamePrefix='flood_risk_tak',
    region=tak.geometry(),
    scale=30,
    maxPixels=1e13
)

task.start()

In [81]:
task.status()

{'state': 'READY',
 'description': 'FloodRisk_Tak',
 'priority': 100,
 'creation_timestamp_ms': 1775064284533,
 'update_timestamp_ms': 1775064295399,
 'start_timestamp_ms': 0,
 'task_type': 'EXPORT_IMAGE',
 'id': 'LBFG5XJEAFJR5Z5W3CDOSUTO',
 'name': 'projects/ee-napaporn3401/operations/LBFG5XJEAFJR5Z5W3CDOSUTO'}

In [83]:
water = ee.Image("JRC/GSW1_4/GlobalSurfaceWater") \
          .select("occurrence") \
          .clip(tak)

water_binary = water.gt(50)
model_binary = flood_risk.gt(0.6)

combined = model_binary.multiply(2).add(water_binary)

hist = combined.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=tak,
    scale=1000,
    maxPixels=1e13
)

result = hist.getInfo()

print(result)

{'total_precipitation_sum': {'0': 18.098039215686228, '1': 144.537254901961}}


In [84]:
{'constant': {'0': ..., '1': ..., '2': ..., '3': ...}}

{'constant': {'0': Ellipsis, '1': Ellipsis, '2': Ellipsis, '3': Ellipsis}}

In [103]:
Map.addLayer(rain_n, {'min':0, 'max':1}, "rain_n")
Map.addLayer(slope_n, {'min':0, 'max':1}, "slope_n")
Map.addLayer(elev_n, {'min':0, 'max':1}, "elev_n")
Map.addLayer(dist_n, {'min':0, 'max':1}, "dist_n")
Map.addLayer(lc_n, {'min':0, 'max':1}, "lc_n")
Map

Map(bottom=750.0, center=[83.01553947297903, 240.22705078125003], controls=(WidgetControl(options=['position',…